# Medical Invoice Parser - Test Notebook

This notebook tests the `MedicalInvoiceParser` for extracting structured data from medical invoice PDFs.

**Purpose**: Validate extraction accuracy and generate JSON output for upstream agentic framework.

**Scope**: 
- Fields directly extractable from PDF (deterministic)
- Fields requiring inference are flagged but not populated (handled by upstream LLM)

In [ ]:
import sys
import json
from pathlib import Path

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from parsers.medical_invoice_parser import (
    MedicalInvoiceParser,
    parse_invoice,           # Convenience function for single file
    parse_invoice_folder     # Convenience function for batch processing
)

## 1. Initialize Parser

In [ ]:
parser = MedicalInvoiceParser()

## 2. Test Invoice 1 - City Osteopathy (Physiotherapy)

**Expected**:
- Provider: City Osteopathy & Physiotherapy Two Private Limited
- Total: SGD 185.30
- Diagnosis: Not present (requires HITL)

In [ ]:
invoice1_path = Path.cwd().parent / 'resources' / 'statements' / 'medical_invoice_statements' / 'invoice1.pdf'
print(f"Parsing: {invoice1_path.name}")
print(f"File exists: {invoice1_path.exists()}")

In [ ]:
result1 = parser.parse(str(invoice1_path))

print(f"Success: {result1.success}")
print(f"Confidence: {result1.confidence:.0%}")
print(f"\nExtracted Data:")
print(json.dumps(result1.data, indent=2))

In [ ]:
# Show fields requiring upstream inference
if result1.requires_review:
    print("Fields requiring inference (to be handled by upstream agentic framework):")
    for field in result1.requires_review:
        print(f"  - {field}")

## 3. Test Invoice 2 - OneDoctors Family Clinic (GP Visit)

**Expected**:
- Provider: ONEDOCTORS FAMILY CLINIC  
- Total: SGD 45.78
- Diagnosis: "Acute upper respiratory infection" (present, needs mapping to C32)

In [ ]:
# Reset parser for fresh state
parser = MedicalInvoiceParser()

invoice2_path = Path.cwd().parent / 'resources' / 'statements' / 'medical_invoice_statements' / 'invoice2.pdf'
result2 = parser.parse(str(invoice2_path))

print(f"Success: {result2.success}")
print(f"Confidence: {result2.confidence:.0%}")
print(f"\nExtracted Data:")
print(json.dumps(result2.data, indent=2))

In [ ]:
# Show fields requiring upstream inference
if result2.requires_review:
    print("Fields requiring inference (to be handled by upstream agentic framework):")
    for field in result2.requires_review:
        print(f"  - {field}")

## 4. Export JSON for Upstream Framework

Generate clean JSON output suitable for the agentic framework to process.

In [ ]:
# The parser now has built-in JSON export methods!
# Option 1: Use MedicalInvoiceResult.to_json() method
# Option 2: Use parse_invoice() convenience function

# Using built-in method on result object:
print("Invoice 1 - Using result.to_json():")
print(result1.to_json_string(source_file="invoice1.pdf"))

In [ ]:
# Using the convenience function (recommended for simple use cases):
invoice1_path = Path.cwd().parent / 'resources' / 'statements' / 'medical_invoice_statements' / 'invoice1.pdf'
output1 = parse_invoice(str(invoice1_path))

print("Invoice 1 - Using parse_invoice() convenience function:")
print(json.dumps(output1, indent=2))

In [ ]:
# Invoice 2
invoice2_path = Path.cwd().parent / 'resources' / 'statements' / 'medical_invoice_statements' / 'invoice2.pdf'
output2 = parse_invoice(str(invoice2_path))

print("Invoice 2 - JSON for upstream framework:")
print(json.dumps(output2, indent=2))

## 5. Batch Processing Example

Process all invoices in the medical_invoice_statements folder.

In [ ]:
# Use the built-in batch processing function
invoice_folder = Path.cwd().parent / 'resources' / 'statements' / 'medical_invoice_statements'

print(f"Processing folder: {invoice_folder}")
print("-" * 50)

all_results = parse_invoice_folder(str(invoice_folder))

# Display summary
for result in all_results:
    filename = result['metadata']['source_file']
    success = result['metadata']['success']
    confidence = result['metadata']['confidence']
    total = result['extracted'].get('paymentAmount', 'N/A')
    
    status = "OK" if success else "FAILED"
    print(f"[{status}] {filename}: SGD {total} (confidence: {confidence:.0%})")

In [ ]:
# Save batch results to JSON file
output_path = Path.cwd().parent / 'extracted_data' / 'medical_invoices_extracted.json'
output_path.parent.mkdir(exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print(f"\nBatch results saved to: {output_path}")

## 6. Validation Summary

Quick validation of extracted amounts against expected values.

In [ ]:
# Expected values from invoices
expected = {
    "invoice1.pdf": {
        "invoiceNumber": "INV-2601000859",
        "paymentAmount": 185.30,
        "gstAmount": 15.30,
        "subtotal": 170.00,
        "visitDate": "2026-01-10"
    },
    "invoice2.pdf": {
        "invoiceNumber": "H101494",
        "paymentAmount": 45.78,
        "gstAmount": 3.78,
        "subtotal": 42.00,
        "visitDate": "2025-04-19"
    }
}

print("Validation Results:")
print("=" * 60)

for result in all_results:
    filename = result['metadata']['source_file']
    extracted = result['extracted']
    exp = expected.get(filename, {})
    
    print(f"\n{filename}:")
    
    for field, exp_value in exp.items():
        ext_value = extracted.get(field)
        
        # Handle float comparison
        if isinstance(exp_value, float) and isinstance(ext_value, float):
            match = abs(exp_value - ext_value) < 0.01
        else:
            match = exp_value == ext_value
        
        status = "PASS" if match else "FAIL"
        print(f"  {field}: {ext_value} (expected: {exp_value}) [{status}]")

---

## Summary

The `MedicalInvoiceParser` successfully extracts:

| Field | Status |
|-------|--------|
| invoiceNumber | Extracted |
| visitDate | Extracted (normalized to YYYY-MM-DD) |
| providerName | Extracted |
| patientName | Extracted |
| subtotal | Extracted or calculated |
| gstAmount | Extracted |
| paymentAmount | Extracted |
| lineItems | Extracted |
| diagnosisRaw | Extracted (when present) |

**Fields for upstream inference**:
- `benefitCategory` - from provider type
- `benefitType` - from provider keywords  
- `diagnosisCode` - from diagnosisRaw text mapping